## **Lab Exercise #8**
- Name: Michael Kenny
- Date Started: 9/16/2025
- Date Completed: 9/24/2025
- Class: CSC-180-02
- Professor: Dr. Mamoun Abu-Samaha

**Assignment Objective:**
- Convert 7 into using Tkinter for GUI.

**Tasks:**
- Use Tkinter for GUI.
    - Add functionality for the GUI to use a built in formula for calculating price of car, then use our own predicted model and compare the results.
    -  Attempt to create a button to swap between OHE and dummy variables. This will require two different models and saving/loading them depending on the choice.

### **1. Reading CSV File into Dataframe**

In [1]:
import pandas as pd

df = pd.read_csv("./carprices4.csv")
df.head()

,Car Model,Mileage,Sell Price($),Age(yrs)
0,BMW X5,69000,18000,6
1,BMW X5,35000,34000,3
2,BMW X5,57000,26100,5
3,BMW X5,22500,40000,2
4,BMW X5,46000,31500,4


We already know that the data resembles a linear structure in both comparisons to Mileage vs. Price and Age vs. Price. Therefore there is no need to graph them again, and we can continue to cleaning data.

### **2. Creating the Dummies Model**

Here I will be splitting my code into two sections, one for the work to create and split the data and train the model for the dummy variable approach, then again for the One-Hot-Encoding approach.

#### **Dummy Variables**

In [2]:
dummies = pd.get_dummies(df['Car Model'], dtype=int)
dummies.head()

,Audi A5,BMW X5,Mercedez Benz C class
0,0,1,0
1,0,1,0
2,0,1,0
3,0,1,0
4,0,1,0


Start by creating dummy variable columns based on the car model.

#### **Merging and Cleaning Dataframe**

In [3]:
merged_dummies = pd.concat([df, dummies], axis='columns')
final_dummies = merged_dummies.drop("Car Model", axis='columns')
final_dummies.head()

,Mileage,Sell Price($),Age(yrs),Audi A5,BMW X5,Mercedez Benz C class
0,69000,18000,6,0,1,0
1,35000,34000,3,0,1,0
2,57000,26100,5,0,1,0
3,22500,40000,2,0,1,0
4,46000,31500,4,0,1,0


Merge the new dummy columns with the original data, then drop the original car model column so we have cleaner looking data.

In [4]:
X_dummies = final_dummies.drop(["Sell Price($)", "Mercedez Benz C class"], axis='columns')
X_dummies.head()

,Mileage,Age(yrs),Audi A5,BMW X5
0,69000,6,0,1
1,35000,3,0,1
2,57000,5,0,1
3,22500,2,0,1
4,46000,4,0,1


Drop the sell price columns and the last dummy variable column (dummy trap) to form the X input for this model.

In [5]:
y_dummies = final_dummies["Sell Price($)"]
y_dummies.head()

0    18000
1    34000
2    26100
3    40000
4    31500
Name: Sell Price($), dtype: int64

Select only the sell price from the final dataframe to be our output/target.

#### **Splitting Data**

In [6]:
from sklearn.model_selection import train_test_split

X_train_dummies, X_test_dummies, y_train_dummies, y_test_dummies = train_test_split(X_dummies, y_dummies, test_size=0.2)

Split the data into train and test in order to train on a part of the original dataset to try and mitigate over fitting.

#### **Training and Saving Model**

In [7]:
from sklearn.linear_model import LinearRegression

reg_model_dummies = LinearRegression()
reg_model_dummies.fit(X_train_dummies, y_train_dummies)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


Train the linear regression model for the dummies approach.

In [8]:
import joblib

joblib.dump(reg_model_dummies, 'model_dummies')

['model_dummies']

Save the model as 'model_dummies' for use later in the Tkinter implementation.

### **3. Creating the OHE Model**

#### **One-Hot-Encoding**

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

dfle = df
dfle['Car Model LE'] = le.fit_transform(df['Car Model'])
dfle.head()

,Car Model,Mileage,Sell Price($),Age(yrs),Car Model LE
0,BMW X5,69000,18000,6,1
1,BMW X5,35000,34000,3,1
2,BMW X5,57000,26100,5,1
3,BMW X5,22500,40000,2,1
4,BMW X5,46000,31500,4,1


#### **Label Encoding**

Use label encoding, even if it is not necessarily needed with the most recent version of sklearn's OneHotEncoder, to create the column.

In [10]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)

car_model_ohe = ohe.fit_transform(dfle[['Car Model LE']])
ohe_columns = ['Audi OHE', 'BMW OHE', 'Mercedez OHE']
ohe_df = pd.DataFrame(car_model_ohe, columns=ohe_columns)
ohe_df.head()

,Audi OHE,BMW OHE,Mercedez OHE
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,0.0,1.0,0.0
3,0.0,1.0,0.0
4,0.0,1.0,0.0


Next we use one hot encoding to transform the label encoded column into the OHE version. This creates a new dataframe with 3 columns that needs to be merged with the original data.

#### **Merging and Cleaning Dataframes**

In [11]:
merged_ohe = pd.concat([dfle, ohe_df], axis='columns')
merged_ohe.head()

,Car Model,Mileage,Sell Price($),Age(yrs),Car Model LE,Audi OHE,BMW OHE,Mercedez OHE
0,BMW X5,69000,18000,6,1,0.0,1.0,0.0
1,BMW X5,35000,34000,3,1,0.0,1.0,0.0
2,BMW X5,57000,26100,5,1,0.0,1.0,0.0
3,BMW X5,22500,40000,2,1,0.0,1.0,0.0
4,BMW X5,46000,31500,4,1,0.0,1.0,0.0


Merge the new OHE columns with the original data.

In [12]:
final_ohe = merged_ohe.drop(['Car Model', 'Car Model LE', 'Mercedez OHE'], axis='columns')
final_ohe.head()

,Mileage,Sell Price($),Age(yrs),Audi OHE,BMW OHE
0,69000,18000,6,0.0,1.0
1,35000,34000,3,0.0,1.0
2,57000,26100,5,0.0,1.0
3,22500,40000,2,0.0,1.0
4,46000,31500,4,0.0,1.0


Drop all the columns that are not required for the in and outputs, this includes the original car model column, label encoded column, and the extra column for OHE.

In [13]:
X_ohe = final_ohe.drop('Sell Price($)', axis='columns')
X_ohe.head()

,Mileage,Age(yrs),Audi OHE,BMW OHE
0,69000,6,0.0,1.0
1,35000,3,0.0,1.0
2,57000,5,0.0,1.0
3,22500,2,0.0,1.0
4,46000,4,0.0,1.0


Drop the sell price column from the cleaned dataframe to contain all of the necessary inputs.

In [14]:
y_ohe = final_ohe['Sell Price($)']
y_ohe.head()

0    18000
1    34000
2    26100
3    40000
4    31500
Name: Sell Price($), dtype: int64

Select the price column from the final dataframe to be the output/target.

#### **Splitting Data**

In [15]:
X_train_ohe, X_test_ohe, y_train_ohe, y_test_ohe = train_test_split(X_ohe, y_ohe, test_size=0.2)

Split the data just like with the dummy variables approach, we do this still to mitigate over fitting.

#### **Training and Saving Model**

In [16]:
from sklearn.linear_model import LinearRegression

reg_model_ohe = LinearRegression()
reg_model_ohe.fit(X_train_ohe, y_train_ohe)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


Train the linear regression model for the OHE approach.

In [17]:
joblib.dump(reg_model_ohe, 'model_ohe')

['model_ohe']

Save the model using joblib for use later in Tkinter.

### **4. Conclusion**

Using Tkinter we can create a new window via Python that we can use as an interface for our model predictions. Using the professor's provided code template, I adjusted and added to the code to accommodate for both the dummies and OHE models as well as the already setup mileage, age, and car model sections. The radio button defaults to the dummy variable model, but can be swapped between that or the OHE model.

One interesting implemetation was for the radio button in the parameters section, the common implementation is to use the command parameter and call some kind of function after the button is activated. I opted to do something different and did not use the command parameter but instead just had the calculate_price method call the method for the radio button selected in order to return the value of the selected model (that was set via the buttons). This way I can setup a simple switch case to choose the right model, similarly to how I chose the correct car.